# 短期记忆

## InMemorySaver

具体步骤如下：
### 1. 导入 CheckPointer 的内存版实现：

In [ ]:
from aiohttp.web_middlewares import middleware
# langchain 提供的 checkpointer 的默认实现，基于内存存储
from langgraph.checkpoint.memory import InMemorySaver

### 2.创建智能体，设置checkpointer：

In [ ]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent

# 创建智能体时指定 checkpointer, langchain 会自动帮我们管理历史会话记忆
agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()
)

### 3.发起调用时，指定 thread_id

- a.第一次调用，告知AI一些信息

In [ ]:
from langchain.messages import HumanMessage

# 设定 thread_id，作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

# 第一次调用，告知AI一些信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好！我叫 heima，我喜欢猫猫")]},
    config  # 调用时添加 thread_id ，区分不同的会话
)

print(response)

- b.第二次调用，询问Ai

In [ ]:
# 第二次调用，询问Ai我的信息，这次带上 thread_id，唤起记忆
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢的动物是？")]},
    config   # 调用时添加 thread_id
)

print(response)

注意：
目前我们使用的checkpointer是基于内存的InMemorySaver，LangChain也提供了很多持久化存储的checkpointer，例如：

- SqlLiteSaver ：基于sqlite存储
- PostgresSaver ：基于Postgres存储
- CosmosDBSaver ：使用Azure Cosmos DB的实现

具体可以查看文档：

https://docs.langchain.com/oss/python/langgraph/persistence#checkpointer-libraries

## 持久化 Memory

LangChain也提供了很多持久化存储的checkpointer，例如：
- SqlLiteSaver ：基于sqlite存储
- PostgresSaver ：基于Postgres存储
- CosmosDBSaver ：使用Azure Cosmos DB的实现

我们以SqlLiteSaver 为例来讲解如何自定义Memory存储方案。

### 首先，安装对应依赖：
`pip install langgraph-checkpoint-sqlite`

`uv add langgraph-checkpoint-sqlite`

### 导入依赖，并初始化 sqlite-checkpointer：

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 初始化 SqliteSaver
checkpointer = SqliteSaver(sqlite3.connect("checkpoint.db", check_same_thread=False))

# 自动建表
checkpointer.setup()

### 创建智能体，指定 checkpointer

In [ ]:
# 创建 agent
agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer
)

# 记忆管理策略

## 总结消息

用法：

### 初始化 SummarizationMiddleware 和 checkpointer

  注意这里SummarizationMiddleware的参数（详细内容参考官网链接：[summarization](https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization)）：

  - model：会话摘要时要使用的模型
  - trigger：会话摘要的触发时机，有三种设置：
    - fraction (float): 模型上下文大小的比例（0-1）
    - tokens (int): 令牌数量
    - messages (int): 消息数量
  - keep：是指触发摘要后要保留的消息
    - fraction (float): 要保留的消息占模型上下文大小的比例（0-1）
    - tokens (int): 要保留的消息的令牌数量
    - messages (int): 要保留的消息数量


In [ ]:
# 加载环境变量
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 初始化 checkpoiinter
checkpointer = InMemorySaver()
# 初始化中间件
middleware = SummarizationMiddleware(
    model="deepseek-chat",
    trigger=("messages", 3),
    keep=("messages", 1)
)

### 创建 Agent, 设置middleware 和 checkpointer

In [ ]:
agent = create_agent(
    model="deepseek-chat",
    middleware=[middleware],
    checkpointer=checkpointer
)

### 调用 Agent

In [ ]:
config: RunnableConfig = {"configurable": {"thread_id": "thread_1"}}

# 制造长会话历史
agent.invoke({"messages": "你好，我是 heima"}, config)
agent.invoke({"messages": "我最喜欢的运动是乒乓球"}, config)
agent.invoke({"messages": "我最喜欢的动物是猫🐱"}, config)

# 测试
final_response = agent.invoke({"messages": "你还记得我吗?"}, config)

for message in final_response['messages']:
    message.pretty_print()
